# EditPro AI Studio: realistic clips and pictures on Kaggle's free GPU

Works with both kinds of **EditPro Kaggle pack**:
- **Movie pack** (Movie Maker): one short realistic video clip per scene with the free, open **LTX-Video** model
  (Lightricks, LTX-Video open-weights licence). Your scene picture (if you gave one) becomes the first frame.
- **Picture pack** (AI Image Generator), and movie scenes without a picture: pictures first, with **RealVisXL V5** (realistic) or
  **DreamShaper XL** (semi-realistic 3D / AI art), both SDXL, openrail++ licence. With face photos, **IP-Adapter Plus-Face** (Apache 2.0)
  keeps the faces (several people: person 1 on the left, then the next one to the right). Movie scenes then become clips from these pictures.

**Before you press Run all:**
1. Right side → **Add Input → Upload** → choose `editpro-kaggle-pack.zip` (from EditPro).
2. Right side → **Settings → Accelerator → GPU T4 x2** (or GPU P100), and **Internet → On** (needs a phone-verified Kaggle account).
3. Menu **Run → Run all**. The first time downloads the model (about 10 GB, several minutes). Then each clip takes a few minutes.
4. When it says **Finished**, open the **Output** panel (right side) and download the zip it names:
   `editpro-clips.zip` → EditPro Movie Maker → **📥 Add clips from Kaggle**;
   `editpro-pictures.zip` → EditPro AI Image Generator → **📥 Add pictures from Kaggle**.

If the session stops halfway, just press Run all again: finished clips and pictures are kept and skipped.

In [ ]:
# Settings you may change
STEPS = 30          # quality steps (more = better but slower; 20-50)
FPS = 24
MODEL = "Lightricks/LTX-Video"
PICTURE_MODELS = {"photo": "SG161222/RealVisXL_V5.0", "semi": "Lykon/dreamshaper-xl-1-0", "ai": "Lykon/dreamshaper-xl-1-0"}
PICTURE_STEPS = 30
FACE_STRENGTH = 0.7  # how strongly the face photos are followed (0.4-1.0): higher = more alike, less free pose

In [ ]:
# 1) Tools (about 2 minutes)
!pip install -q -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg

In [ ]:
# 2) Find the EditPro pack and the graphics card
import glob, json, os, shutil, time, zipfile
import torch

packs = glob.glob("/kaggle/input/**/jobs.json", recursive=True)
if not packs:
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        zipfile.ZipFile(z).extractall("/kaggle/working/pack")
    packs = glob.glob("/kaggle/working/pack/**/jobs.json", recursive=True)
assert packs, 'EditPro pack not found. Right side: "Add Input" -> Upload -> editpro-kaggle-pack.zip'
PACK = os.path.dirname(packs[0])
jobs = json.load(open(packs[0], encoding="utf-8"))
assert jobs.get("app") == "EditPro" and jobs.get("version") == 1, "This is not an EditPro pack."
scenes = jobs.get("scenes", [])
pictures = jobs.get("pictures", [])
print(len(scenes), "scenes and", len(pictures), "pictures to make")
assert torch.cuda.is_available(), "No GPU. Right side: Settings -> Accelerator -> GPU T4 x2 (or P100)"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 3) Pictures first: picture packs (AI Image Generator) and movie scenes that have no picture yet.
#    Finished ones are skipped, so you can run this again.
from PIL import Image
from diffusers.utils import load_image
PICS = "/kaggle/working/pictures"
SCENE_PICS = "/kaggle/working/scene-pictures"
os.makedirs(PICS, exist_ok=True); os.makedirs(SCENE_PICS, exist_ok=True)
jobs_pics = [dict(p, out=f"{PICS}/pic-{p['n']}-{p['id']}.png") for p in pictures]
for s in scenes:
    if not s.get("image") and s.get("picture_prompt"):
        out = f"{SCENE_PICS}/scene-{s['n']}-{s['id']}.png"
        s["image"] = out  # the clip below starts from this picture
        jobs_pics.append(dict(n=s["n"], id=s["id"], prompt=s["picture_prompt"], negative=s.get("negative", ""), width=s.get("picture_width", 1024),
                              height=s.get("picture_height", 1024), seed=s.get("seed", s["n"]), faces=s.get("faces", []), look=s.get("look", "photo"), out=out))
for p in jobs_pics:
    if os.path.exists(p["out"]):
        print("already made:", os.path.basename(p["out"]))
todo = [p for p in jobs_pics if not os.path.exists(p["out"])]
if todo:
    from diffusers import StableDiffusionXLPipeline
    pp, loaded = None, None
    # Grouped by model and by "with faces", so each model is loaded once.
    for p in sorted(todo, key=lambda x: (PICTURE_MODELS.get(x.get("look", "photo"), PICTURE_MODELS["photo"]), len(x.get("faces", [])) > 0)):
        faces = [load_image(os.path.join(PACK, f)).convert("RGB") for f in p.get("faces", [])]
        model = PICTURE_MODELS.get(p.get("look", "photo"), PICTURE_MODELS["photo"])
        want = (model, bool(faces))
        if want != loaded:
            pp = None
            torch.cuda.empty_cache()
            extra = {}
            if faces:
                from transformers import CLIPVisionModelWithProjection
                print("loading the face model…")
                extra["image_encoder"] = CLIPVisionModelWithProjection.from_pretrained("h94/IP-Adapter", subfolder="models/image_encoder", torch_dtype=torch.float16)
            print("loading the picture model", model, "…")
            pp = StableDiffusionXLPipeline.from_pretrained(model, torch_dtype=torch.float16, variant="fp16", **extra)
            if faces:
                pp.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter-plus-face_sdxl_vit-h.safetensors")
            pp.enable_model_cpu_offload()
            loaded = want
        w, h = p["width"], p["height"]
        args = dict(prompt=p["prompt"], negative_prompt=p.get("negative", ""), width=w, height=h, num_inference_steps=PICTURE_STEPS,
                    guidance_scale=5.0, generator=torch.Generator("cpu").manual_seed(int(p.get("seed") or 0)))
        if len(faces) == 1:
            pp.set_ip_adapter_scale(FACE_STRENGTH)
            args["ip_adapter_image"] = faces[0]
        elif len(faces) > 1:
            # Several people: face 1 is used in the left strip of the picture, face 2 in the next strip, and so on.
            from diffusers.image_processor import IPAdapterMaskProcessor
            n = len(faces)
            masks = []
            for k in range(n):
                m = Image.new("L", (w, h), 0); m.paste(255, (k * w // n, 0, (k + 1) * w // n, h)); masks.append(m)
            m = IPAdapterMaskProcessor().preprocess(masks, height=h, width=w)
            pp.set_ip_adapter_scale([[FACE_STRENGTH] * n])
            args["ip_adapter_image"] = [faces]
            args["cross_attention_kwargs"] = {"ip_adapter_masks": [m.reshape(1, m.shape[0], m.shape[2], m.shape[3])]}
        t0 = time.time()
        img = pp(**args).images[0]
        img.save(p["out"] + ".part.png")
        os.replace(p["out"] + ".part.png", p["out"])
        print(f"picture {os.path.basename(p['out'])}: done in {time.time() - t0:.0f} s")
    pp = None
    torch.cuda.empty_cache()

In [ ]:
# 4) Make the clips (finished ones are skipped, so you can run this again)
from diffusers import LTXImageToVideoPipeline, LTXPipeline
from diffusers.utils import export_to_video, load_image
import shutil as _sh

OUT = "/kaggle/working/clips"
os.makedirs(OUT, exist_ok=True)
NEG = "worst quality, inconsistent motion, blurry, jittery, distorted, deformed face, extra fingers, text, watermark"

def frames_for(seconds):
    n = int(round(seconds * FPS / 8)) * 8 + 1  # LTX needs 8k+1 frames
    return max(25, min(161, n))

pipe, kind_loaded = None, None
order = sorted(scenes, key=lambda s: s.get("image") is None)  # picture scenes first: one model switch at most
for s in order:
    out = f"{OUT}/scene-{s['n']}-{s['id']}.mp4"
    if os.path.exists(out):
        print("already made:", os.path.basename(out))
        continue
    kind = "i2v" if s.get("image") else "t2v"
    if kind != kind_loaded:
        pipe = None
        torch.cuda.empty_cache()
        cls = LTXImageToVideoPipeline if kind == "i2v" else LTXPipeline
        print("loading the", "picture-to-video" if kind == "i2v" else "text-to-video", "model…")
        pipe = cls.from_pretrained(MODEL, torch_dtype=torch.bfloat16)
        pipe.enable_model_cpu_offload()
        kind_loaded = kind
    args = dict(prompt=s["prompt"], negative_prompt=NEG, width=s["width"], height=s["height"],
                num_frames=frames_for(s["seconds"]), num_inference_steps=STEPS)
    if kind == "i2v":
        args["image"] = load_image(os.path.join(PACK, s["image"])).convert("RGB").resize((s["width"], s["height"]))
    t0 = time.time()
    video = pipe(**args).frames[0]
    export_to_video(video, out + ".part.mp4", fps=FPS)
    os.replace(out + ".part.mp4", out)
    print(f"scene {s['n']}: done in {time.time() - t0:.0f} s")
for s in scenes:  # the scene pictures go back to EditPro too (shown in the storyboard)
    if (s.get("image") or "").startswith(SCENE_PICS) and os.path.exists(s["image"]):
        _sh.copy(s["image"], f"{OUT}/scene-{s['n']}-{s['id']}.png")
pipe = None
torch.cuda.empty_cache()

In [ ]:
# 5) One file to download
if scenes:
    shutil.make_archive("/kaggle/working/editpro-clips", "zip", OUT)
    print("Finished:", len([f for f in os.listdir(OUT) if f.endswith(".mp4")]), "clips. Download editpro-clips.zip from the Output panel (right side).")
if pictures:
    shutil.make_archive("/kaggle/working/editpro-pictures", "zip", PICS)
    print("Finished:", len([f for f in os.listdir(PICS) if f.endswith(".png")]), "pictures. Download editpro-pictures.zip from the Output panel (right side).")